In [ ]:
Oracle AI Data Platform v1.0

Copyright © 2025, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# MongoDB Atlas Connector Samples

Read-only ingestion samples for MongoDB Atlas using the `aidp-connector-mongodb` package and the MongoDB Spark Connector, Spark format `mongodb`. Replace all placeholders before running a sample.

## Prerequisites

1. **Connector jar and package.** In a clone of [arbisoft/oracle-aidp-connectors](https://github.com/arbisoft/oracle-aidp-connectors), in `src/aidp_connector_mongodb`, run:
   ```
   python build_connector_jar.py
   uv build
   ```
   The first downloads the MongoDB Spark Connector and its four driver jars from Maven Central, checks each SHA-256 and merges them into `dist/mongo-spark-connector-bundle_2.12-10.7.0.jar`. The second builds `dist/aidp_connector_mongodb-<version>-py3-none-any.whl`.
2. **Cluster libraries.** Upload both files to a workspace folder. Install each from the cluster's **Library** tab → **Install Library** → **Workspace**, one at a time, then restart the cluster.
3. **Credential.** Create a **Secret Token** credential in the Credential Store with key `uri`, holding a read-only user's connection string:
   ```
   mongodb+srv://<user>:<password>@<cluster>.mongodb.net/
   ```
4. **Network.** Add the cluster's outbound IP to the Atlas project's **IP access list**.

## Configuration

The collection to read and the Delta table to load it into.

In [ ]:
from aidp_connector_mongodb import format_summary, parse_config, read_uri, run
from aidp_connector_mongodb.reader import read_collection, resolve_srv

config = parse_config({
    "mongodb": {
        "credential_name": "<CREDENTIAL_NAME>",  # Secret Token credential with key "uri"
        "database": "<DATABASE>",
        "collection": "<COLLECTION>",
    },
    "target": {"catalog": "<CATALOG>", "schema": "<SCHEMA>", "table": "<TABLE_NAME>"},
    "sync": {
        "watermark_field": "<DATE_FIELD>",  # must hold BSON Dates; None = full re-read every run
        "fields": [],                        # top-level fields to load; [] = every field
        "string_fields": [],                 # fields whose type differs between documents
    },
})

## Connection URI

Reads the connection string from the Credential Store and resolves it on the driver, because AIDP executors cannot.

In [ ]:
uri = resolve_srv(spark, read_uri(config.mongodb, aidputils.secrets.get))
print("connection URI resolved (not shown)")

## Ingestion Sample

Reads the collection with a schema inferred from a sample and widened, so values wider than the sample are not read as null.

In [ ]:
mongodb_df = read_collection(spark, uri, config.mongodb.database, config.mongodb.collection,
                             fields=config.sync.columns, string_fields=config.sync.string_fields,
                             sample_size=config.sync.sample_size)

mongodb_df.show()

## Incremental Load into Delta

Loads the whole collection on the first run, then only documents with a newer `watermark_field`, merged on `_id`. A full refresh reloads the whole collection and overwrites the table, which also removes documents deleted in MongoDB.

In [ ]:
summary = run(spark, config, uri)  # run(spark, config, uri, "full") for a full refresh

print(format_summary(summary))

## Connector Options

| Parameter name | Valid values | Mandatory | Description |
| --- | --- | --- | --- |
| `mongodb.credential_name` | Credential Store entry | Y | Entry holding the connection string. |
| `mongodb.credential_key` | Key name | N | Key inside that entry. Default `uri`. |
| `mongodb.database` | Database name | Y | Database to read from. |
| `mongodb.collection` | Collection name | Y | Collection to read from. |
| `target.catalog` | Existing catalog | Y | Target catalog. |
| `target.schema` | Schema name | Y | Target schema. Created if missing. |
| `target.table` | Table name | Y | Target table. |
| `sync.watermark_field` | Date field, or `None` | N | Field used to read only changed documents. `None` re-reads the whole collection. |
| `sync.fields` | List of field names | N | Fields to load. Empty loads every field. |
| `sync.string_fields` | List of field names | N | Fields read as text because their type differs between documents. |
| `sync.sample_size` | Positive integer | N | Documents sampled to infer the schema on the first run. Default `10000`. |